# Rating Prediction Model
Training Linear Regression and Random Forest Regressor for rating prediction (1-5 stars)

In [ ]:
!pip install scikit-learn pandas numpy nltk joblib -q

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import joblib
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer
import re

nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)

In [ ]:
# Create training data
data = {
    'review': [
        'Excellent product, amazing quality',
        'Perfect, exactly what I wanted',
        'Very good, satisfied',
        'Good enough for the price',
        'Average, nothing special',
        'Not so good',
        'Disappointing quality',
        'Terrible, do not buy',
    ],
    'rating': [5.0, 5.0, 4.5, 4.0, 3.0, 2.5, 2.0, 1.0]
}

df = pd.DataFrame(data)
print('Rating distribution:')
print(df['rating'].describe())

In [ ]:
# Preprocessing
stemmer = PorterStemmer()
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'[^a-z0-9\s]', '', text)
    tokens = word_tokenize(text)
    tokens = [stemmer.stem(token) for token in tokens if token not in stop_words]
    return ' '.join(tokens)

df['review_processed'] = df['review'].apply(preprocess_text)

In [ ]:
# Feature Extraction
vectorizer = TfidfVectorizer(max_features=5000, min_df=1)
X = vectorizer.fit_transform(df['review_processed'])
y = df['rating']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
# Train Linear Regression
print('Training Linear Regression...')
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

y_pred_lr = lr_model.predict(X_test)

print('\n=== Linear Regression Performance ===')
print(f'MAE: {mean_absolute_error(y_test, y_pred_lr):.4f}')
print(f'RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_lr)):.4f}')
print(f'R² Score: {r2_score(y_test, y_pred_lr):.4f}')

In [ ]:
# Train Random Forest Regressor
print('Training Random Forest Regressor...')
rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)

print('\n=== Random Forest Regressor Performance ===')
print(f'MAE: {mean_absolute_error(y_test, y_pred_rf):.4f}')
print(f'RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_rf)):.4f}')
print(f'R² Score: {r2_score(y_test, y_pred_rf):.4f}')

In [ ]:
# Save model
import os
os.makedirs('/content/models', exist_ok=True)

joblib.dump(rf_model, '/content/models/rating_model.pkl')
print('✓ Rating prediction model saved')

In [ ]:
# Test predictions
test_reviews = [
    'Excellent quality product',
    'Average product',
    'Terrible experience',
]

print('=== Test Predictions ===\n')

for review in test_reviews:
    processed = preprocess_text(review)
    X_review = vectorizer.transform([processed])
    
    prediction = rf_model.predict(X_review)[0]
    # Clamp to 1-5 range
    rating = max(1, min(5, prediction))
    
    print(f'Review: {review}')
    print(f'Predicted Rating: {rating:.2f}/5.0\n')